<a href="https://colab.research.google.com/github/yashb98/90Days_Machine_learinng/blob/main/Synapse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install Dependencies
!pip install -q streamlit # -q for "quiet"
!pip install -q langchain langchain-openai llama-index openai faiss-cpu sentence-transformers pandas python-dotenv
!pip install -q openai-whisper

# Install ffmpeg for audio processing
!apt-get install -y -qq ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 k

In [2]:
# Cell 2: Create Directories
import os

os.makedirs("data/synthea_csv", exist_ok=True)
os.makedirs("data/ehr", exist_ok=True)
os.makedirs("data/golden_path", exist_ok=True)

print("Created directory structure:")
!ls -R data

Created directory structure:
data:
ehr  golden_path  synthea_csv

data/ehr:

data/golden_path:

data/synthea_csv:


In [3]:
!pip install datasets

In [5]:
# Cell 2: Create Directories & Download MTS-Dialog
import os
from datasets import load_dataset

print("Creating directory structure...")
os.makedirs("data/ehr", exist_ok=True)
os.makedirs("data/golden_path", exist_ok=True)
!ls -R data

print("\nDownloading MTS-Dialog dataset from Hugging Face...")
# This dataset has 'train', 'validation', 'test' splits. We'll use 'train'.
try:
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')
    print("\nMTS-Dialog dataset loaded successfully.")
    print(f"Total samples: {len(mts_dataset)}")

    # Let's inspect the first sample
    print("\n--- Sample 1 ---")
    print(f"[DIALOGUE]:\n{mts_dataset[0]['dialogue']}")
    print(f"\n[NOTE]:\n{mts_dataset[0]['note']}")
    print("------------------")

except Exception as e:
    print(f"Error loading dataset: {e}")


Creating directory structure...
data:
ehr  golden_path  synthea_csv

data/ehr:

data/golden_path:

data/synthea_csv:


MTS-Dialog dataset loaded successfully.
Total samples: 1301

--- Sample 1 ---
[DIALOGUE]:
Doctor: What brings you back into the clinic today, miss? 
Patient: I came in for a refill of my blood pressure medicine. 
Doctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones.  Have you noticed any changes or do you have any concerns regarding these issues?  
Patient: No. 
Doctor: Have you had any fever or chills, cough, congestion, nausea, vomiting, chest pain, chest pressure?
Patient: No.  
Doctor: Great. Also, for our records, how old are you and what race do you identify yourself as?
Patient: I am seventy six years old and identify as a white female.
Error loading dataset: 'note'


In [6]:
# Cell 3: Mount Google Drive
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

# --- VERIFY YOUR PATH ---
# This command lists the files in your 'csv' folder.
# If this command fails, your folder isn't at 'My Drive/csv'.
# Adjust the path as needed.
print("\nVerifying access to your Synthea files...")
!ls -lh /content/drive/MyDrive/csv

Mounting Google Drive...
Mounted at /content/drive

Verifying access to your Synthea files...
total 539M
-rw------- 1 root root 140K Nov 19  2021 allergies.csv
-rw------- 1 root root 740K Nov 19  2021 careplans.csv
-rw------- 1 root root  41M Nov 19  2021 claims.csv
-rw------- 1 root root 296M Nov 19  2021 claims_transactions.csv
-rw------- 1 root root 4.9M Nov 19  2021 conditions.csv
-rw------- 1 root root  20K Nov 19  2021 devices.csv
-rw------- 1 root root  19M Nov 19  2021 encounters.csv
-rw------- 1 root root  51M Nov 19  2021 imaging_studies.csv
-rw------- 1 root root 2.4M Nov 19  2021 immunizations.csv
-rw------- 1 root root  14M Nov 19  2021 medications.csv
-rw------- 1 root root  88M Nov 19  2021 observations.csv
-rw------- 1 root root 153K Nov 19  2021 organizations.csv
-rw------- 1 root root 330K Nov 19  2021 patients.csv
-rw------- 1 root root 2.2K Nov 19  2021 payers.csv
-rw------- 1 root root 8.7M Nov 19  2021 payer_transitions.csv
-rw------- 1 root root  15M Nov 19  2021

In [9]:
# Cell 4.5: Full Column Name Diagnostic
import pandas as pd
from pathlib import Path
import os

# Path to your Synthea CSVs in Google Drive
CSV_DIR = Path("/content/drive/MyDrive/csv")

# List of all 18 CSV files from your screenshot
csv_files_list = [
    "allergies.csv",
    "careplans.csv",
    "claims_transactions.csv",
    "claims.csv",
    "conditions.csv",
    "devices.csv",
    "encounters.csv",
    "imaging_studies.csv",
    "immunizations.csv",
    "medications.csv",
    "observations.csv",
    "organizations.csv",
    "patients.csv",
    "payer_transitions.csv",
    "payers.csv",
    "procedures.csv",
    "providers.csv",
    "supplies.csv"
]

print(f"--- Reading all column headers from {CSV_DIR} ---")
print("This will check all 18 files...\n")

missing_files = []

# Loop through each file
for file_name in csv_files_list:
    file_path = CSV_DIR / file_name

    # Check if file exists
    if not file_path.exists():
        print(f"!!! WARNING: File not found: {file_name} !!!\n")
        missing_files.append(file_name)
        continue

    # Read only the header row (nrows=0) to get columns
    try:
        df_header = pd.read_csv(file_path, nrows=0)

        print(f"--- Columns in {file_name} ---")
        print(df_header.columns.tolist())
        print("--------------------------------" + "-" * len(file_name) + "\n")

    except pd.errors.EmptyDataError:
        print(f"--- {file_name} is empty ---")
        print("[]")
        print("-----------------------" + "-" * len(file_name) + "\n")
    except Exception as e:
        print(f"!!! Error reading {file_name}: {e} !!!\n")

if missing_files:
    print(f"\nSummary: Could not find the following files: {missing_files}")
else:
    print("\nSummary: All 18 files were found and headers were read successfully.")

--- Reading all column headers from /content/drive/MyDrive/csv ---
This will check all 18 files...

--- Columns in allergies.csv ---
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'SYSTEM', 'DESCRIPTION', 'TYPE', 'CATEGORY', 'REACTION1', 'DESCRIPTION1', 'SEVERITY1', 'REACTION2', 'DESCRIPTION2', 'SEVERITY2']
---------------------------------------------

--- Columns in careplans.csv ---
['Id', 'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'REASONCODE', 'REASONDESCRIPTION']
---------------------------------------------

--- Columns in claims_transactions.csv ---
['ID', 'CLAIMID', 'CHARGEID', 'PATIENTID', 'TYPE', 'AMOUNT', 'METHOD', 'FROMDATE', 'TODATE', 'PLACEOFSERVICE', 'PROCEDURECODE', 'MODIFIER1', 'MODIFIER2', 'DIAGNOSISREF1', 'DIAGNOSISREF2', 'DIAGNOSISREF3', 'DIAGNOSISREF4', 'UNITS', 'DEPARTMENTID', 'NOTES', 'UNITAMOUNT', 'TRANSFEROUTID', 'TRANSFERTYPE', 'PAYMENTS', 'ADJUSTMENTS', 'TRANSFERS', 'OUTSTANDING', 'APPOINTMENTID', 'LINENOTE', 'PATIENTINSURANCEID', 'FE

In [10]:
# Cell 4 (Corrected): Define the ENHANCED Synthea processing script
import pandas as pd
from pathlib import Path
import os

# Path to your Synthea CSVs in Google Drive
CSV_DIR = Path("/content/drive/MyDrive/csv")
OUTPUT_DIR = Path("data/ehr")

def process_synthea_data_enhanced():
    """
    Reads multiple Synthea CSVs from Google Drive and creates one rich
    .txt file per patient in the Colab 'data/ehr/' directory.

    (Version 2 - Corrected DATE/START key error)
    """
    print(f"Reading CSVs from: {CSV_DIR}")

    if not CSV_DIR.exists():
        print(f"Error: Directory not found: {CSV_DIR}")
        return

    try:
        patients = pd.read_csv(CSV_DIR / "patients.csv")
        meds = pd.read_csv(CSV_DIR / "medications.csv")
        conditions = pd.read_csv(CSV_DIR / "conditions.csv")
        allergies = pd.read_csv(CSV_DIR / "allergies.csv")
        procedures = pd.read_csv(CSV_DIR / "procedures.csv")
        encounters = pd.read_csv(CSV_DIR / "encounters.csv")
        observations = pd.read_csv(CSV_DIR / "observations.csv")
    except FileNotFoundError as e:
        print(f"Error loading file: {e}")
        print(f"Please ensure all CSV files (patients, meds, conditions, etc.) are in your folder: {CSV_DIR}")
        return
    except Exception as e:
        print(f"An error occurred: {e}")
        return

    # Create output directory
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Processing {len(patients)} patients...")

    # Process each patient
    for _, patient in patients.iterrows():
        patient_id = patient["Id"]

        # 1. Demographics
        patient_info = [
            f"Patient ID: {patient_id}",
            f"Name: {patient['FIRST']} {patient['LAST']}",
            f"Gender: {patient['GENDER']}",
            f"Birthdate: {patient['BIRTHDATE']}",
            f"Address: {patient.get('ADDRESS', 'N/A')}",
            f"Marital Status: {patient.get('MARITAL', 'N/A')}",
        ]

        # 2. Allergies
        patient_allergies = allergies[allergies["PATIENT"] == patient_id]
        allergy_list = [f"- {desc}" for desc in patient_allergies["DESCRIPTION"].unique()]

        # 3. Active Conditions
        patient_conditions = conditions[conditions["PATIENT"] == patient_id]
        condition_list = [f"- {desc}" for desc in patient_conditions["DESCRIPTION"].unique()]

        # 4. Current Medications
        patient_meds = meds[meds["PATIENT"] == patient_id]
        med_list = [f"- {desc}" for desc in patient_meds["DESCRIPTION"].unique()]

        # 5. Past Procedures
        patient_procs = procedures[procedures["PATIENT"] == patient_id]
        # FIX: Changed 'DATE' to 'START'
        proc_list = [f"- {row['DESCRIPTION']} (Date: {row['START']})" for _, row in patient_procs.iterrows()]

        # 6. Recent Encounters
        # FIX: Changed 'DATE' to 'START' for sorting
        patient_encs = encounters[encounters["PATIENT"] == patient_id].sort_values('START', ascending=False)
        # FIX: Changed 'row['DATE']' to 'row['START']'
        enc_list = [f"- {row['START']}: {row['DESCRIPTION']}" for _, row in patient_encs.head(5).iterrows()]

        # 7. Recent Observations (Vitals/Labs)
        # NO FIX NEEDED: 'DATE' column exists in observations.csv
        patient_obs = observations[observations["PATIENT"] == patient_id].sort_values('DATE', ascending=False)
        obs_list = [f"- {row['DATE']} {row['DESCRIPTION']}: {row['VALUE']} {row.get('UNITS', '')}" for _, row in patient_obs.head(10).iterrows()]

        # Assemble the text file content
        content = f"== PATIENT RECORD: {patient['FIRST']} {patient['LAST']} (ID: {patient_id}) ==\n\n"
        content += "== Demographics ==\n" + "\n".join(patient_info) + "\n\n"
        content += "== Allergies ==\n" + ("\n".join(allergy_list) if allergy_list else "None on record.") + "\n\n"
        content += "== Active Conditions / Problem List ==\n" + ("\n".join(condition_list) if condition_list else "None on record.") + "\n\n"
        content += "== Current Medications ==\n" + ("\n".join(med_list) if med_list else "None on record.") + "\n\n"
        content += "== Past Procedures ==\n" + ("\n".join(proc_list) if proc_list else "None on record.") + "\n\n"
        content += "== Recent Encounters (Last 5) ==\n" + ("\n".join(enc_list) if enc_list else "None on record.") + "\n\n"
        content += "== Recent Observations (Last 10) ==\n" + ("\n".join(obs_list) if obs_list else "None on record.") + "\n"

        # Write to file
        output_filename = OUTPUT_DIR / f"patient_{patient_id}.txt"
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(content)

    print(f"\nSuccessfully processed and saved {len(patients)} patient records to {OUTPUT_DIR}")
    print(f"Total files in {OUTPUT_DIR}: {len(list(OUTPUT_DIR.glob('*.txt')))}")

In [11]:
# Cell 5: Run the processing
process_synthea_data_enhanced()

# Check the output - let's find a random file and print it
print("\n--- Sample Enhanced EHR File ---")
!ls data/ehr | head -n 1 | xargs -I {} head -n 25 data/ehr/{}

Reading CSVs from: /content/drive/MyDrive/csv
Processing 1163 patients...

Successfully processed and saved 1163 patient records to data/ehr
Total files in data/ehr: 1163

--- Sample Enhanced EHR File ---
== PATIENT RECORD: Fatima244 Sauer652 (ID: 00126cb9-8460-4747-e302-c3609684531e) ==

== Demographics ==
Patient ID: 00126cb9-8460-4747-e302-c3609684531e
Name: Fatima244 Sauer652
Gender: F
Birthdate: 1987-05-30
Address: 142 Streich Trail
Marital Status: M

== Allergies ==
None on record.

== Active Conditions / Problem List ==
- Hypertension
- Received higher education (finding)
- Unemployed (finding)
- Social isolation (finding)
- Normal pregnancy
- Stress (finding)
- Acute viral pharyngitis (disorder)
- Limited social contact (finding)
- Preeclampsia
- Unhealthy alcohol drinking behavior (finding)
- Victim of intimate partner abuse (finding)


In [13]:
# Cell 5.5: Diagnostic Cell - Check MTS-Dialog Columns
from datasets import load_dataset

try:
    # Ensure dataset is loaded
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')

    # Print all column names
    print("--- Columns in MTS-Dialog Dataset ---")
    print(mts_dataset.column_names)
    print("-------------------------------------")

    # Print the first sample to see the structure
    print("\n--- First Sample Data ---")
    print(mts_dataset[0])
    print("---------------------------")

except Exception as e:
    print(f"An error occurred: {e}")

--- Columns in MTS-Dialog Dataset ---
['ID', 'section_header', 'section_text', 'dialogue']
-------------------------------------

--- First Sample Data ---
{'ID': 0, 'section_header': 'GENHX', 'section_text': 'Symptoms: no fever, no chills, no cough, no congestion, no nausea, no vomiting, no chest pain, no chest pressure.\nDiagnosis: hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis, kidney stones\nHistory of Patient: 76-year-old white female, presents to the clinic today originally for hypertension and a med check, followed by Dr. Kumar, issues stable\nPlan of Action: N/A', 'dialogue': 'Doctor: What brings you back into the clinic today, miss? \nPatient: I came in for a refill of my blood pressure medicine. \nDoctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones.  Have you noticed any changes or do you have any concerns regarding these 

In [20]:
# Cell 6 (Corrected - Final): Prepare Golden Path Data
from datasets import load_dataset
import textwrap

# Load dataset (if not already loaded)
try:
    mts_dataset
except NameError:
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')

# --- PICK YOUR GOLDEN PATH SAMPLE ---
# We'll use sample index 5. You can change this index if you want.
SAMPLE_INDEX = 5
# -----------------------------------

golden_sample = mts_dataset[SAMPLE_INDEX]
golden_dialogue = golden_sample['dialogue']

# --- THIS IS THE FIX ---
# The column is 'section_text', not 'note' or 'summary'
golden_note = golden_sample['section_text']
# ---------------------

# Save these to our golden_path folder for later
with open("data/golden_path/conversation.txt", "w", encoding="utf-8") as f:
    f.write(golden_dialogue)

with open("data/golden_path/ground_truth_note.txt", "w", encoding="utf-8") as f:
    f.write(golden_note)

print("--- PLEASE RECORD THIS DIALOGUE AS 'conversation.mp3' ---")
print(f"--- (Sample {SAMPLE_INDEX}) ---")
print("\n".join(textwrap.wrap(golden_dialogue, 80)))
print("\n---------------------------------------------------------")
print("Saved text to data/golden_path/conversation.txt")
print("Saved note to data/golden_path/ground_truth_note.txt")

--- PLEASE RECORD THIS DIALOGUE AS 'conversation.mp3' ---
--- (Sample 5) ---
Doctor: How's your asthma since you started using your inhaler again?  Patient:
Much better. I don't know why I didn't take it with me everywhere I went.
Doctor: It's important to carry it with you, especially during times where
you're exercising or walking more than usual.  Patient: Yeah. I think I've
learned my lesson.  Doctor: Besides asthma, do you have any other medical
problems?

---------------------------------------------------------
Saved text to data/golden_path/conversation.txt
Saved note to data/golden_path/ground_truth_note.txt


In [21]:
# Cell 7: Upload 'Golden Path' Audio
from google.colab import files
import os

print("Please upload your 'Golden Path' audio file (conversation.mp3)")
uploaded = files.upload()

# Move uploaded files to the correct directory
for fn in uploaded.keys():
    if fn.lower().endswith((".mp3", ".wav", ".m4a")):
        os.rename(fn, "data/golden_path/Conversation2.mp3")
        print(f"Renamed '{fn}' to 'conversation2.mp3'")
    else:
        print(f"Warning: Uploaded file '{fn}' was not an expected audio file. Trying to rename anyway.")
        os.rename(fn, "data/golden_path/conversation2.mp3")


print("\nFile uploaded to data/golden_path/:")
!ls data/golden_path

Please upload your 'Golden Path' audio file (conversation.mp3)


Saving Conversation2.mp3 to Conversation2.mp3
Renamed 'Conversation2.mp3' to 'conversation2.mp3'

File uploaded to data/golden_path/:
Conversation2.mp3  conversation.txt	  transcript_from_audio.txt
conversation.mp3   ground_truth_note.txt


In [25]:
# Cell 8: Define and Run ASR (Whisper)
import whisper
import time
from pathlib import Path

# Caching the model load
_model = None

def get_whisper_model():
    """Loads and caches the Whisper model."""
    global _model
    if _model is None:
        print("Loading Whisper model (small.en)... This may take a moment.")
        # Using "small.en" for a good balance of speed and accuracy on Colab GPU
        _model = whisper.load_model("small.en")
        print("Whisper model loaded.")
    return _model

def transcribe_audio(filepath: str) -> str:
    """Transcribes an audio file using Whisper."""
    print(f"Starting transcription for: {filepath}")
    model = get_whisper_model()

    start_time = time.time()
    try:
        # We are in Colab with a GPU, so this should be fast
        result = model.transcribe(filepath)
        end_time = time.time()
        duration = end_time - start_time
        print(f"Transcription finished in {duration:.2f} seconds.")
        return result["text"]

    except Exception as e:
        print(f"Error during transcription: {e}")
        return ""

# --- Now, let's run it ---
audio_file = "data/golden_path/Conversation2.mp3"
transcript_file = "data/golden_path/transcript_from_audio.txt"

if not os.path.exists(audio_file):
    print(f"Error: Audio file not found at {audio_file}")
    print("Please re-run Cell 7 to upload your file.")
else:
    # Run the transcription
    transcript = transcribe_audio(audio_file)

    if transcript:
        print("\n--- WHISPER TRANSCRIPT ---")
        print(transcript)
        print("----------------------------")

        # Save to a file for Day 2
        with open(transcript_file, "w", encoding="utf-8") as f:
            f.write(transcript)
        print(f"Transcript saved to {transcript_file}")

Starting transcription for: data/golden_path/Conversation2.mp3
Loading Whisper model (small.en)... This may take a moment.
Whisper model loaded.


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription finished in 25.88 seconds.

--- WHISPER TRANSCRIPT ---
 Here. How's your asthma since you started using inhaler again? Patient, much better. I don't know why I didn't take it with me everywhere. When? Doctor, it's important to carry it with you. Especially during times where you are exercising or walking more than usual. Patient, yeah, I think I have learned my lesson. Doctor, besides asthma, do you have any other medical problems?
----------------------------
Transcript saved to data/golden_path/transcript_from_audio.txt
